# 1. Building a Complete REST API

This notebook covers building a production-ready REST API with:
- Complete CRUD operations
- Input validation
- Error handling
- Pagination
- Search and filtering

# 2. Project Structure

```
flask_api/
├── app/
│   ├── __init__.py         # App factory
│   ├── routes/
│   │   ├── __init__.py
│   │   ├── users.py        # User routes
│   │   └── products.py     # Product routes
│   ├── models/
│   │   └── __init__.py     # Data models
│   ├── services/
│   │   └── __init__.py     # Business logic
│   └── utils/
│       ├── validators.py   # Validation helpers
│       └── responses.py    # Response helpers
├── config.py               # Configuration
├── run.py                  # Entry point
└── requirements.txt        # Dependencies
```

# 3. Response Helpers

In [ ]:
# utils/responses.py - Standardized API responses

responses_code = '''
from flask import jsonify

def api_response(data=None, message=None, status=200):
    """Create a success API response."""
    response = {
        "success": True,
    }
    if data is not None:
        response["data"] = data
    if message:
        response["message"] = message
    return jsonify(response), status


def error_response(message, status=400, errors=None):
    """Create an error API response."""
    response = {
        "success": False,
        "error": {
            "message": message
        }
    }
    if errors:
        response["error"]["details"] = errors
    return jsonify(response), status


def paginated_response(items, page, per_page, total):
    """Create a paginated API response."""
    total_pages = (total + per_page - 1) // per_page
    return jsonify({
        "success": True,
        "data": items,
        "meta": {
            "page": page,
            "per_page": per_page,
            "total": total,
            "total_pages": total_pages,
            "has_next": page < total_pages,
            "has_prev": page > 1
        }
    })
'''

print("Response Helpers:")
print(responses_code)

# 4. Input Validation

In [ ]:
# utils/validators.py - Input validation helpers

validators_code = '''
import re

class ValidationError(Exception):
    """Custom validation error."""
    def __init__(self, message, errors=None):
        self.message = message
        self.errors = errors or []


def validate_email(email):
    """Validate email format."""
    pattern = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$"
    return re.match(pattern, email) is not None


def validate_required(data, fields):
    """
    Validate that required fields are present.
    Returns list of missing fields.
    """
    missing = []
    for field in fields:
        if field not in data or data[field] is None or data[field] == "":
            missing.append(field)
    return missing


def validate_user_data(data, is_update=False):
    """
    Validate user input data.
    Returns dict with 'valid' bool and 'errors' list.
    """
    errors = []
    
    # Required fields (only for create)
    if not is_update:
        required = ["name", "email"]
        missing = validate_required(data, required)
        if missing:
            errors.append(f"Missing required fields: {\', \'.join(missing)}")
    
    # Validate email format
    if "email" in data and data["email"]:
        if not validate_email(data["email"]):
            errors.append("Invalid email format")
    
    # Validate name length
    if "name" in data and data["name"]:
        if len(data["name"]) < 2:
            errors.append("Name must be at least 2 characters")
        if len(data["name"]) > 100:
            errors.append("Name must be less than 100 characters")
    
    # Validate age if provided
    if "age" in data and data["age"] is not None:
        if not isinstance(data["age"], int) or data["age"] < 0 or data["age"] > 150:
            errors.append("Age must be a valid number between 0 and 150")
    
    return {
        "valid": len(errors) == 0,
        "errors": errors
    }
'''

print("Validation Helpers:")
print(validators_code)

# 5. Complete CRUD API

In [ ]:
# Complete CRUD API example
# Save as app.py and run with: python app.py

crud_api_code = '''
from flask import Flask, request, jsonify
from flask_cors import CORS
from datetime import datetime
import re

app = Flask(__name__)
CORS(app)

# ==========================================
# IN-MEMORY DATABASE
# ==========================================

users_db = [
    {"id": 1, "name": "John Doe", "email": "john@email.com", "age": 30, "active": True, "created_at": "2024-01-15T10:00:00"},
    {"id": 2, "name": "Jane Smith", "email": "jane@email.com", "age": 25, "active": True, "created_at": "2024-01-16T11:00:00"},
    {"id": 3, "name": "Bob Wilson", "email": "bob@email.com", "age": 35, "active": False, "created_at": "2024-01-17T12:00:00"},
]
next_id = 4

# ==========================================
# HELPER FUNCTIONS
# ==========================================

def find_user(user_id):
    return next((u for u in users_db if u["id"] == user_id), None)

def validate_email(email):
    pattern = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\\.[a-zA-Z0-9-.]+$"
    return re.match(pattern, email) is not None

def success_response(data=None, message=None, status=200):
    response = {"success": True}
    if data is not None:
        response["data"] = data
    if message:
        response["message"] = message
    return jsonify(response), status

def error_response(message, status=400, errors=None):
    response = {"success": False, "error": {"message": message}}
    if errors:
        response["error"]["details"] = errors
    return jsonify(response), status

def paginated_response(items, page, per_page, total):
    total_pages = (total + per_page - 1) // per_page if total > 0 else 0
    return jsonify({
        "success": True,
        "data": items,
        "meta": {
            "page": page,
            "per_page": per_page,
            "total": total,
            "total_pages": total_pages,
            "has_next": page < total_pages,
            "has_prev": page > 1
        }
    })

# ==========================================
# ERROR HANDLERS
# ==========================================

@app.errorhandler(404)
def not_found(e):
    return error_response("Resource not found", 404)

@app.errorhandler(400)
def bad_request(e):
    return error_response(str(e.description) if hasattr(e, "description") else "Bad request", 400)

@app.errorhandler(500)
def server_error(e):
    return error_response("Internal server error", 500)

# ==========================================
# ROUTES
# ==========================================

@app.route("/")
def home():
    return {
        "message": "Users REST API",
        "version": "1.0",
        "endpoints": {
            "list": "GET /api/users",
            "get": "GET /api/users/<id>",
            "create": "POST /api/users",
            "update": "PUT /api/users/<id>",
            "delete": "DELETE /api/users/<id>"
        }
    }


# ==========================================
# GET ALL USERS (with pagination, filtering, sorting)
# ==========================================
@app.route("/api/users", methods=["GET"])
def get_users():
    # Pagination parameters
    page = request.args.get("page", 1, type=int)
    per_page = request.args.get("per_page", 10, type=int)
    per_page = min(per_page, 100)  # Max 100 per page
    
    # Filter parameters
    name_filter = request.args.get("name")
    active_filter = request.args.get("active")
    
    # Sort parameters
    sort_by = request.args.get("sort_by", "id")
    sort_order = request.args.get("sort_order", "asc")
    
    # Apply filters
    filtered_users = users_db.copy()
    
    if name_filter:
        filtered_users = [u for u in filtered_users if name_filter.lower() in u["name"].lower()]
    
    if active_filter is not None:
        is_active = active_filter.lower() == "true"
        filtered_users = [u for u in filtered_users if u["active"] == is_active]
    
    # Apply sorting
    if sort_by in ["id", "name", "email", "age", "created_at"]:
        reverse = sort_order.lower() == "desc"
        filtered_users.sort(key=lambda x: x.get(sort_by, ""), reverse=reverse)
    
    # Apply pagination
    total = len(filtered_users)
    start = (page - 1) * per_page
    end = start + per_page
    paginated_users = filtered_users[start:end]
    
    return paginated_response(paginated_users, page, per_page, total)


# ==========================================
# GET SINGLE USER
# ==========================================
@app.route("/api/users/<int:user_id>", methods=["GET"])
def get_user(user_id):
    user = find_user(user_id)
    if not user:
        return error_response(f"User with ID {user_id} not found", 404)
    return success_response(user)


# ==========================================
# CREATE USER
# ==========================================
@app.route("/api/users", methods=["POST"])
def create_user():
    global next_id
    
    # Parse JSON data
    data = request.get_json()
    if not data:
        return error_response("Request body must be JSON")
    
    # Validation
    errors = []
    
    if not data.get("name"):
        errors.append("Name is required")
    elif len(data["name"]) < 2:
        errors.append("Name must be at least 2 characters")
    
    if not data.get("email"):
        errors.append("Email is required")
    elif not validate_email(data["email"]):
        errors.append("Invalid email format")
    elif any(u["email"] == data["email"] for u in users_db):
        errors.append("Email already exists")
    
    if "age" in data and data["age"] is not None:
        if not isinstance(data["age"], int) or data["age"] < 0:
            errors.append("Age must be a positive integer")
    
    if errors:
        return error_response("Validation failed", 422, errors)
    
    # Create user
    new_user = {
        "id": next_id,
        "name": data["name"].strip(),
        "email": data["email"].strip().lower(),
        "age": data.get("age"),
        "active": data.get("active", True),
        "created_at": datetime.now().isoformat()
    }
    users_db.append(new_user)
    next_id += 1
    
    return success_response(new_user, "User created successfully", 201)


# ==========================================
# UPDATE USER (Full update - PUT)
# ==========================================
@app.route("/api/users/<int:user_id>", methods=["PUT"])
def update_user(user_id):
    user = find_user(user_id)
    if not user:
        return error_response(f"User with ID {user_id} not found", 404)
    
    data = request.get_json()
    if not data:
        return error_response("Request body must be JSON")
    
    # Validation
    errors = []
    
    if "name" in data:
        if not data["name"] or len(data["name"]) < 2:
            errors.append("Name must be at least 2 characters")
    
    if "email" in data:
        if not validate_email(data["email"]):
            errors.append("Invalid email format")
        elif any(u["email"] == data["email"] and u["id"] != user_id for u in users_db):
            errors.append("Email already exists")
    
    if "age" in data and data["age"] is not None:
        if not isinstance(data["age"], int) or data["age"] < 0:
            errors.append("Age must be a positive integer")
    
    if errors:
        return error_response("Validation failed", 422, errors)
    
    # Update user
    if "name" in data:
        user["name"] = data["name"].strip()
    if "email" in data:
        user["email"] = data["email"].strip().lower()
    if "age" in data:
        user["age"] = data["age"]
    if "active" in data:
        user["active"] = bool(data["active"])
    
    return success_response(user, "User updated successfully")


# ==========================================
# PARTIAL UPDATE (PATCH)
# ==========================================
@app.route("/api/users/<int:user_id>", methods=["PATCH"])
def patch_user(user_id):
    # PATCH can use the same logic as PUT in this case
    return update_user(user_id)


# ==========================================
# DELETE USER
# ==========================================
@app.route("/api/users/<int:user_id>", methods=["DELETE"])
def delete_user(user_id):
    user = find_user(user_id)
    if not user:
        return error_response(f"User with ID {user_id} not found", 404)
    
    users_db.remove(user)
    return success_response(message=f"User {user_id} deleted successfully")


# ==========================================
# BULK OPERATIONS
# ==========================================
@app.route("/api/users/bulk", methods=["DELETE"])
def bulk_delete_users():
    """Delete multiple users by IDs."""
    data = request.get_json()
    if not data or "ids" not in data:
        return error_response("Missing \'ids\' in request body")
    
    ids_to_delete = data["ids"]
    if not isinstance(ids_to_delete, list):
        return error_response("\'ids\' must be an array")
    
    deleted = []
    not_found = []
    
    for user_id in ids_to_delete:
        user = find_user(user_id)
        if user:
            users_db.remove(user)
            deleted.append(user_id)
        else:
            not_found.append(user_id)
    
    return success_response({
        "deleted": deleted,
        "not_found": not_found
    })


# ==========================================
# RUN APP
# ==========================================
if __name__ == "__main__":
    app.run(debug=True, port=5000)
'''

print("Complete CRUD API (save as app.py):")
print("="*60)
print(crud_api_code)

# 6. Testing the API with Python Requests

In [ ]:
# Testing code - run this after starting the Flask server

test_code = '''
import requests
import json

BASE_URL = "http://localhost:5000/api"

def print_response(response, title=""):
    if title:
        print(f"\\n{'='*50}")
        print(title)
        print(f"{'='*50}")
    print(f"Status: {response.status_code}")
    print(f"Response: {json.dumps(response.json(), indent=2)}")

# ==========================================
# TEST CRUD OPERATIONS
# ==========================================

# 1. GET all users
response = requests.get(f"{BASE_URL}/users")
print_response(response, "GET All Users")

# 2. GET with pagination
response = requests.get(f"{BASE_URL}/users?page=1&per_page=2")
print_response(response, "GET Users (Page 1, 2 per page)")

# 3. GET with filtering
response = requests.get(f"{BASE_URL}/users?active=true&name=john")
print_response(response, "GET Users (filtered)")

# 4. GET single user
response = requests.get(f"{BASE_URL}/users/1")
print_response(response, "GET User by ID")

# 5. POST - Create user
new_user = {
    "name": "Alice Brown",
    "email": "alice@email.com",
    "age": 28
}
response = requests.post(f"{BASE_URL}/users", json=new_user)
print_response(response, "POST Create User")
created_id = response.json()["data"]["id"] if response.ok else None

# 6. PUT - Update user
if created_id:
    update_data = {
        "name": "Alice Brown Updated",
        "age": 29
    }
    response = requests.put(f"{BASE_URL}/users/{created_id}", json=update_data)
    print_response(response, "PUT Update User")

# 7. DELETE user
if created_id:
    response = requests.delete(f"{BASE_URL}/users/{created_id}")
    print_response(response, "DELETE User")

# 8. Test validation error
invalid_user = {
    "name": "A",  # Too short
    "email": "invalid-email"  # Invalid format
}
response = requests.post(f"{BASE_URL}/users", json=invalid_user)
print_response(response, "POST Invalid User (Validation Error)")

# 9. Test 404
response = requests.get(f"{BASE_URL}/users/9999")
print_response(response, "GET Non-existent User (404)")
'''

print("Test Code (run after starting Flask server):")
print("="*60)
print(test_code)

# 7. curl Commands for Testing

In [ ]:
curl_commands = '''
# ==========================================
# CURL COMMANDS FOR TESTING
# ==========================================

# GET all users
curl http://localhost:5000/api/users

# GET with pagination
curl "http://localhost:5000/api/users?page=1&per_page=2"

# GET with filtering and sorting
curl "http://localhost:5000/api/users?active=true&sort_by=name&sort_order=asc"

# GET single user
curl http://localhost:5000/api/users/1

# POST - Create user
curl -X POST http://localhost:5000/api/users \
  -H "Content-Type: application/json" \
  -d '{"name": "New User", "email": "new@email.com", "age": 25}'

# PUT - Update user
curl -X PUT http://localhost:5000/api/users/1 \
  -H "Content-Type: application/json" \
  -d '{"name": "Updated Name", "age": 31}'

# PATCH - Partial update
curl -X PATCH http://localhost:5000/api/users/1 \
  -H "Content-Type: application/json" \
  -d '{"active": false}'

# DELETE user
curl -X DELETE http://localhost:5000/api/users/1

# Bulk delete
curl -X DELETE http://localhost:5000/api/users/bulk \
  -H "Content-Type: application/json" \
  -d '{"ids": [1, 2, 3]}'
'''

print(curl_commands)

# 8. Summary

## REST API Best Practices Implemented

| Feature | Implementation |
|---------|----------------|
| **Consistent Response Format** | `{"success": bool, "data": ..., "error": ...}` |
| **Proper Status Codes** | 200 OK, 201 Created, 404 Not Found, 422 Validation Error |
| **Input Validation** | Required fields, format validation, error messages |
| **Pagination** | `?page=1&per_page=10` with meta info |
| **Filtering** | `?name=john&active=true` |
| **Sorting** | `?sort_by=name&sort_order=asc` |
| **Error Handling** | Global error handlers for 400, 404, 500 |
| **CORS** | Enabled for cross-origin requests |

## HTTP Methods Used

| Method | Route | Action |
|--------|-------|--------|
| GET | `/api/users` | List all (with pagination) |
| GET | `/api/users/:id` | Get one |
| POST | `/api/users` | Create |
| PUT | `/api/users/:id` | Full update |
| PATCH | `/api/users/:id` | Partial update |
| DELETE | `/api/users/:id` | Delete one |
| DELETE | `/api/users/bulk` | Delete many |